In [ ]:
! pip install marker-pdf
! pip install pypdf
! pip install google
! pip install vllm

In [ ]:
import os
import subprocess
import time
import urllib.request
import urllib.error
from pathlib import Path
from google.colab import drive
from pypdf import PdfReader, PdfWriter

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
from marker.output import text_from_rendered

In [ ]:
os.environ["SURYA_INFERENCE_BACKEND"] = "vllm"
os.environ["SURYA_INFERENCE_URL"] = "http://localhost:8000/v1"
os.environ["SURYA_INFERENCE_KEEP_ALIVE"] = "1"

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "true"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_DEVICE"] = "cuda"

In [ ]:
drive.mount('/content/drive')
INPUT_DIR = "/content/drive/MyDrive/learningagentbooks"
OUTPUT_DIR = "/content/drive/MyDrive/learningagentmarkdown"

In [ ]:
def shorten_pdf(input_path, output_path, pages):
  reader = PdfReader(input_path)
  writer = PdfWriter()

  for page_num in range(pages):
    writer.add_page(reader.pages[page_num])

  with open(output_path, "wb") as f:
    writer.write(f)

def check_file_names():
    os.makedirs("/content/drive/MyDrive/learningagentbooks", exist_ok=True)
    os.makedirs("/content/drive/MyDrive/learningagentmarkdown", exist_ok=True)

    visible_files = os.listdir("/content/drive/MyDrive/learningagentbooks")
    visible_files = visible_files + os.listdir("/content/drive/MyDrive/learningagentmarkdown")
    if len(visible_files) == 0:
        print("The folder is EMPTY!")
    else:
        for file in visible_files:
            print(f"Found: {file}")

In [ ]:
SURYA_SERVER_URL = "http://localhost:8000/v1"
SURYA_MODEL = "datalab-to/surya-ocr-2"
 
_surya_process = None
 
def _surya_server_is_up(timeout: float = 2.0) -> bool:
    try:
        urllib.request.urlopen(f"{SURYA_SERVER_URL}/models", timeout=timeout)
        return True
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        return False

def _ensure_surya_server(max_wait_seconds: int = 300) -> None:
    """Start the vllm/surya server as a plain background process (no
    Docker) if nothing is already listening on it, then block until it's
    actually ready to accept requests.
    """
    global _surya_process
 
    if _surya_server_is_up():
        return
 
    if _surya_process is None:
        print("Starting surya vllm server (first call only, this can take a "
              "couple of minutes while the model downloads/loads)...")
        _surya_process = subprocess.Popen(
            ["vllm", "serve", SURYA_MODEL, "--port", "8000"]
        )
 
    waited = 0
    while not _surya_server_is_up():
        if _surya_process.poll() is not None:
            raise RuntimeError(
                "surya vllm server process exited before becoming ready. "
                "Check the notebook output above for the actual error."
            )
        if waited >= max_wait_seconds:
            raise TimeoutError(
                f"surya vllm server did not become ready within "
                f"{max_wait_seconds}s."
            )
        time.sleep(5)
        waited += 5
 
    print("surya server is ready.")

def convert_with_marker(file_name: str) -> str:
    _ensure_surya_server()

    input_path = os.path.join(INPUT_DIR, file_name)
    output_path = os.path.join(OUTPUT_DIR, Path(file_name).stem + ".md")

    config = {
        "output_format": "markdown",
        "force_ocr": True,
        "paginate_output": True,
        "disable_image_extraction": True,
    }
    config_parser = ConfigParser(config)

    converter = PdfConverter(
        config=config_parser.generate_config_dict(),
        artifact_dict=create_model_dict(),
        processor_list=config_parser.get_processors(),
        renderer=config_parser.get_renderer(),
        llm_service=config_parser.get_llm_service(),
    )

    rendered = converter(input_path)
    markdown_text, _, _ = text_from_rendered(rendered)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_text)

    return output_path

In [ ]:
input_file_name = "test_snippet.pdf"
if __name__ == "__main__":
    os.makedirs(INPUT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    convert_with_marker(input_file_name)